In [0]:
import yaml, os

config_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "./config.yaml",
)

with open(config_path) as f:
    _cfg = yaml.safe_load(f)

# Expose top-level keys as variables
TOTAL_FILES                   = _cfg["TOTAL_FILES"]
PAGE_SIZE                     = _cfg["PAGE_SIZE"]
NUM_WORKERS                   = _cfg["NUM_WORKERS"]
MANIFEST_TABLE                = _cfg["MANIFEST_TABLE"]
DEST_VOLUME                   = _cfg["DEST_VOLUME"]
MLFLOW_EXPERIMENT             = _cfg["MLFLOW_EXPERIMENT"]

SECRETS_SCOPE                 = _cfg["SECRETS_SCOPE"]
SECRETS_KEY_ACCESS_KEY_ID     = _cfg["SECRETS_KEY_ACCESS_KEY_ID"]
SECRETS_KEY_SECRET_ACCESS_KEY = _cfg["SECRETS_KEY_SECRET_ACCESS_KEY"]
BUCKET                        = _cfg["BUCKET"]
PREFIX                        = _cfg["PREFIX"]
REGION                        = _cfg["REGION"]
ENDPOINT_URL                  = _cfg["ENDPOINT_URL"]


access_key = dbutils.secrets.get(scope=SECRETS_SCOPE, key=SECRETS_KEY_ACCESS_KEY_ID)
secret_key = dbutils.secrets.get(scope=SECRETS_SCOPE, key=SECRETS_KEY_SECRET_ACCESS_KEY)

os.environ["AWS_ACCESS_KEY_ID"]     = access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = secret_key

print(f"Config loaded: {len(_cfg)} keys")
print(f"  ENDPOINT_URL   = {ENDPOINT_URL}")
print(f"  BUCKET         = {BUCKET}")
print(f"  PREFIX         = {PREFIX}")
print(f"  REGION         = {REGION}")
print(f"  MANIFEST_TABLE = {MANIFEST_TABLE}")
print(f"  DEST_VOLUME    = {DEST_VOLUME}")
print(f"  TOTAL_FILES    = {TOTAL_FILES}, PAGE_SIZE = {PAGE_SIZE}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (
  file_id INT,
  source_bucket STRING,
  source_key STRING,
  size_bytes BIGINT,
  dest_filename STRING,
  page_num INT)
""")

In [0]:
import boto3
import pandas as pd
import numpy as np
import time

# 1. Initialize the S3 client pointed to your Isilon S3 endpoint
s3_client = boto3.client(
    's3',
    endpoint_url            = ENDPOINT_URL,
    aws_access_key_id       = access_key,
    aws_secret_access_key   = secret_key,
    verify=True
)

bucket_name = BUCKET

# 2. Use a paginator to get all objects (handles Isilon/S3's 1000-object limit)
paginator = s3_client.get_paginator('list_objects_v2')
page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=PREFIX)

# 3. Iterate through pages and print object keys (file paths)
rows = []
for page in page_iterator:
    if 'Contents' in page:
        for obj in page['Contents']:
            rows.append({
                'source_bucket': bucket_name,
                'source_key': obj['Key'],
                'size_bytes': obj['Size']
            })
df = pd.DataFrame(rows)
df = df.sort_values(by=['size_bytes'], ascending=False)
df = df.head(TOTAL_FILES)
df = df.reset_index(drop=True)

# insert into MANIFEST table
df = spark.createDataFrame(df)
df.write.mode("overwrite").saveAsTable(MANIFEST_TABLE)

In [0]:
df_manifest = spark.table(MANIFEST_TABLE)
print(f"Manifest rows: {df_manifest.count()}")
display(df_manifest)

In [0]:

#spark.sql(f"""UPDATE {MANIFEST_TABLE} SET source_bucket = '{BUCKET}'""")